In [ ]:
# FrozenLake 8x8 Q-learning

import os

import gymnasium as gym
import matplotlib.pyplot as plt
import numpy as np


ALPHA = 0.5
GAMMA = 0.99
EPS_START = 1.0
EPS_MIN = 0.05
EPS_DECAY = 0.9997
EPISODES = 50_000
EXP_EPISODES = 25_000  # Fewer episodes than baseline; each setting uses 3 seeds.
WINDOW = 500

os.makedirs("results", exist_ok=True)


def make_env(render_mode=None):
    return gym.make(
        "FrozenLake-v1",
        map_name="8x8",
        is_slippery=True,
        render_mode=render_mode,
    )


def train(alpha, gamma, decay, episodes, seed=42):
    np.random.seed(seed)
    env = make_env()
    env.reset(seed=seed)
    env.action_space.seed(seed)

    q = np.zeros((env.observation_space.n, env.action_space.n))  # 64 states x 4 actions
    rewards = np.zeros(episodes)
    eps = EPS_START

    for ep in range(episodes):
        s, _ = env.reset()
        done = False
        while not done:
            # Epsilon greedy
            if np.random.random() < eps:
                a = env.action_space.sample()
            else:
                a = np.argmax(q[s])
            s2, r, terminated, truncated, _ = env.step(a)
            done = terminated or truncated

            # No future value once the episode is over.
            if done:
                target = r
            else:
                target = r + gamma * np.max(q[s2])
            q[s, a] += alpha * (target - q[s, a])

            s = s2
            rewards[ep] += r
        eps = max(EPS_MIN, eps * decay)  # Decay once per episode, not per step.

    env.close()
    return q, rewards


def smooth(x):
    # Rewards are just 0 or 1, so this moving average makes the graph readable.
    return np.convolve(x, np.ones(WINDOW) / WINDOW, mode="valid")


def make_plot(results, title, fname):
    plt.figure(figsize=(9, 5))
    for label in results:
        y = smooth(results[label])
        plt.plot(np.arange(WINDOW, WINDOW + len(y)), y, label=label)
    plt.xlabel("episode")
    plt.ylabel("avg reward")
    plt.title(title)
    plt.legend()
    plt.savefig("results/" + fname)
    plt.close()


def evaluate(q, episodes=100):
    env = make_env()
    wins = 0
    total_steps = 0
    for ep in range(episodes):
        s, _ = env.reset()
        done = False
        steps = 0
        while not done:
            a = np.argmax(q[s])  # No exploration here.
            s, r, terminated, truncated, _ = env.step(a)
            steps += 1
            done = terminated or truncated
            wins += r  # r is 1 only if the agent reached the goal.
        total_steps += steps
    env.close()
    return wins / episodes, total_steps / episodes


def record_best_agent(q, seed=2026):
    """Record one greedy episode after training; this does not affect learning."""
    env = gym.wrappers.RecordVideo(
        make_env(render_mode="rgb_array"),
        video_folder="results",
        episode_trigger=lambda episode_number: episode_number == 0,
        name_prefix="frozenlake_agent",
        disable_logger=True,
    )
    s, _ = env.reset(seed=seed)
    done = False
    while not done:
        a = np.argmax(q[s])
        s, _, terminated, truncated, _ = env.step(a)
        done = terminated or truncated
    env.close()
    print("video saved as results/frozenlake_agent-episode-0.mp4")


# Baseline
q_base, r_base = train(ALPHA, GAMMA, EPS_DECAY, EPISODES)
make_plot({"baseline": r_base}, "baseline", "baseline.png")

best_score = -1

# Alpha experiment: average 3 seeds because one run can be noisy.
alpha_results = {}
for a in [0.1, 0.5, 0.9]:
    runs = []
    for seed in range(3):
        q, r = train(a, GAMMA, EPS_DECAY, EXP_EPISODES, seed)
        runs.append(r)
    alpha_results["alpha=" + str(a)] = np.mean(runs, axis=0)
    score = smooth(alpha_results["alpha=" + str(a)])[-1]
    print("alpha", a, score)
    if score > best_score:
        best_score = score
        best = (a, GAMMA, EPS_DECAY)
make_plot(alpha_results, "learning rate", "alpha.png")

# Gamma experiment: same process.
gamma_results = {}
for g in [0.5, 0.9, 0.99]:
    runs = []
    for seed in range(3):
        q, r = train(ALPHA, g, EPS_DECAY, EXP_EPISODES, seed)
        runs.append(r)
    gamma_results["gamma=" + str(g)] = np.mean(runs, axis=0)
    score = smooth(gamma_results["gamma=" + str(g)])[-1]
    print("gamma", g, score)
    if score > best_score:
        best_score = score
        best = (ALPHA, g, EPS_DECAY)
make_plot(gamma_results, "discount factor", "gamma.png")

# Epsilon-decay experiment: the extra hyperparameter.
decay_results = {}
for d in [0.999, 0.9997, 0.9999]:
    runs = []
    for seed in range(3):
        q, r = train(ALPHA, GAMMA, d, EXP_EPISODES, seed)
        runs.append(r)
    decay_results["decay=" + str(d)] = np.mean(runs, axis=0)
    score = smooth(decay_results["decay=" + str(d)])[-1]
    print("decay", d, score)
    if score > best_score:
        best_score = score
        best = (ALPHA, GAMMA, d)
make_plot(decay_results, "epsilon decay", "decay.png")

# Retrain the best setting for the full number of episodes and compare it to baseline.
print("best config (alpha, gamma, decay):", best)
q_best, r_best = train(best[0], best[1], best[2], EPISODES, seed=43)

base_sr, base_steps = evaluate(q_base)
best_sr, best_steps = evaluate(q_best)
print("baseline: success rate", base_sr, "avg steps", base_steps)
print("best: success rate", best_sr, "avg steps", best_steps)

# This is only for the assignment video requirement.
record_best_agent(q_best)
